**A Spark Cluster is a group of computers (machines/nodes) working together to process large amounts of data.Instead of running on a single machine.**

          Cluster
    ┌───────────────┐
    │ Driver Node   │
    └───────┬───────┘
            │
 ┌──────────┼──────────┐
 │          │          │
 ▼          ▼          ▼
Worker 1  Worker 2  Worker 3

In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("User Functions")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
# Reading with Schema
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

emp = spark.read.format("csv").option("header",True).schema(_schema).load('/content/drive/MyDrive/emp.csv')

In [4]:
emp.rdd.getNumPartitions()

1

In [5]:
# Create a function to generate 10% of Salary as Bonus

def bonus(salary):
    return int(salary) * 0.1

In [6]:
# Register as UDF
from pyspark.sql.functions import udf

bonus_udf = udf(bonus)

spark.udf.register("bonus_sql_udf", bonus, "double")

<function __main__.bonus(salary)>

In [7]:
emp.withColumn("bonus", bonus_udf("salary")).show()

+-----------+-------------+-------------+---+------+-------+----------+------+
|employee_id|department_id|         name|age|gender| salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+-------+----------+------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|5000.0|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|4500.0|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|5500.0|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|4800.0|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|6000.0|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|5200.0|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|7000.0|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|5100.0|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|5800.0|
|         10|          104|     Lisa Lee| 27|Female|

In [8]:
# Create new column as bonus using UDF
from pyspark.sql.functions import expr

emp.withColumn("bonus", expr("bonus_sql_udf(salary)")).show()

+-----------+-------------+-------------+---+------+-------+----------+------+
|employee_id|department_id|         name|age|gender| salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+-------+----------+------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|5000.0|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|4500.0|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|5500.0|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|4800.0|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|6000.0|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|5200.0|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|7000.0|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|5100.0|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|5800.0|
|         10|          104|     Lisa Lee| 27|Female|

In [9]:
# Create new column as bonus without UDF

emp.withColumn("bonus", expr("salary * 0.1")).show()

+-----------+-------------+-------------+---+------+-------+----------+------+
|employee_id|department_id|         name|age|gender| salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+-------+----------+------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|5000.0|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|4500.0|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|5500.0|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|4800.0|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|6000.0|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|5200.0|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|7000.0|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|5100.0|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|5800.0|
|         10|          104|     Lisa Lee| 27|Female|

In Apache Spark, a DAG (Directed Acyclic Graph) is the execution plan Spark creates to process your data efficiently.

Breaking down the name

Directed → Operations have a direction (from input → output).

Acyclic → No loops or cycles.

Graph → A collection of nodes and edges representing transformations.

Why DAG?

Spark uses DAGs to:

Optimize execution

Reduce data movement

Combine operations

Execute tasks in parallel

This is one reason Spark is faster than Hadoop MapReduce.

Adaptive Query Execution is a Spark optimization feature that dynamically changes the execution plan at runtime based on actual data statistics, improving performance through techniques like join strategy switching, skew handling, and partition coalescing.

A Broadcast Join is a Spark optimization where a small table is copied (broadcast) to all executors, so Spark can join it with a large table without shuffling the large table across the network.

Partition Coalescing means reducing the number of partitions to avoid having too many small partitions that create unnecessary overhead.

In [10]:
# Disable AQE(Adaptive Query Execution) and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [11]:
# Check default Parallism

spark.sparkContext.defaultParallelism

2

In [12]:
# Create dataframes

df_1 = spark.range(4, 200, 2)
df_2 = spark.range(2, 200, 4)

In [13]:
df_2.rdd.getNumPartitions()

2

In [14]:
# Re-partition data

df_3 = df_1.repartition(5)
df_4 = df_2.repartition(7)

In [15]:
df_4.rdd.getNumPartitions()

7

In [17]:
df_3.show()

+---+
| id|
+---+
| 26|
| 60|
| 46|
| 86|
| 10|
| 68|
| 42|
| 80|
|  4|
| 58|
|104|
|180|
|128|
|106|
|168|
|192|
|152|
|142|
|144|
|162|
+---+
only showing top 20 rows


In [21]:
df_3.count()

98

In [18]:
df_4.show()

+---+
| id|
+---+
| 14|
| 86|
| 42|
|146|
|134|
|142|
|162|
| 74|
| 94|
| 34|
|198|
|182|
|126|
|174|
| 98|
| 10|
| 82|
|122|
|186|
|158|
+---+
only showing top 20 rows


In [22]:
df_4.count()

50

In [19]:
# Join the dataframes

df_joined = df_3.join(df_4, on="id")

In [20]:
df_joined.show()

+---+
| id|
+---+
| 26|
| 54|
| 22|
|198|
|130|
| 34|
|126|
| 50|
| 94|
|110|
|190|
| 98|
|  6|
| 58|
|158|
|178|
|150|
|146|
|174|
|170|
+---+
only showing top 20 rows


In [23]:
df_joined.count()

49

In [24]:
# Get the sum of ids

df_sum = df_joined.selectExpr("sum(id) as total_sum")

In [25]:
df_sum.show()

+---------+
|total_sum|
+---------+
|     4998|
+---------+



In [26]:
# Explain plan

df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#118L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=501]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#118L)])
      +- *(5) Project [id#118L]
         +- *(5) SortMergeJoin [id#118L], [id#119L], Inner
            :- *(2) Sort [id#118L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#118L, 200), ENSURE_REQUIREMENTS, [plan_id=485]
            :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=484]
            :        +- *(1) Range (4, 200, step=2, splits=2)
            +- *(4) Sort [id#119L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#119L, 200), ENSURE_REQUIREMENTS, [plan_id=492]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [plan_id=491]
                     +- *(3) Range (2, 200, step=4, splits=2)




In [31]:
spark.stop()

In [32]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Optimizing Shuffles")
    .master("local[*]")
    .getOrCreate()
)

spark

In [33]:
spark.sparkContext.defaultParallelism

2

In [34]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [37]:
# Read EMP CSV file with 10M records

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load('/content/employee_records.csv')

In [38]:
# Find out avg salary as per dept
from pyspark.sql.functions import avg

emp_avg = emp.groupBy("department_id").agg(avg("salary").alias("avg_sal"))

In [45]:
emp_avg.show()

+-------------+------------------+
|department_id|           avg_sal|
+-------------+------------------+
|           10| 502682.2575766687|
|            5| 504167.9429997006|
|            1|504876.96401242825|
|            3| 504697.6808514883|
|            2| 503563.2174529479|
|            6|504428.12590014644|
|            9| 504945.3055672206|
|            7|504514.38453985273|
|            4| 505419.4963977089|
|            8| 505299.1226286386|
+-------------+------------------+



In [39]:
# Write data for performance Benchmarking

emp_avg.write.format("noop").mode("overwrite").save()

In [40]:
# Check Spark Shuffle Partition setting

spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [43]:
spark.conf.set("spark.sql.shuffle.partitions", 16)

In [44]:
from pyspark.sql.functions import spark_partition_id

emp.withColumn("partition_id", spark_partition_id()).where("partition_id = 0").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|partition_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|           0|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|           0|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|           0|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|           0|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffany

In [46]:
spark.stop()